In [1]:
%cd /home/anw2067/visualnav-transformer/train

import torch
import yaml
import copy
import wandb

from torchvision import transforms
from dreamsim import dreamsim

from diffusers.models import AutoencoderKL

from peva.models import CDiT_models
from peva.diffusion import create_diffusion

from vint_train.data.misc import XSensConstants, XsensSkeleton
from vint_train.data.vint_dataset import ViNT_Nymeria_Dataset
from vint_train.training.nymeria_training_utils import get_action_smpl_torch
from planning.cem import CEMPlanner

from torchvision.utils import save_image

PEVA_CONFIG="/home/anw2067/visualnav-transformer/train/peva/config/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_-64to_64_1_goal_emb_relative_xxl.yaml"
PEVA_CHECKPOINT="/scratch/anw2067/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_cancel_scaler_-64to_64_xxl_280_0180000.pth.tar"

NOMAD_CONFIG = "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/config.yaml"

/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/anw2067/visualnav-transformer/train


/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/wandb/sdk/internal/internal_api.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_peva(peva_config_file, peva_checkpoint, diffusion_steps=250, inference_context_size=15, device='cpu'):
    with open(peva_config_file, 'r') as f:
        peva_config = yaml.safe_load(f)
    num_cond = peva_config['context_size']
    if inference_context_size is None:
        inference_context_size = num_cond
    
    if peva_config.get('full_body', False):
        num_head_joint_cond = 23
    else:
        num_head_joint_cond = 15
        
    model = CDiT_models[peva_config['model']](
        context_size=num_cond, 
        inference_context_size=inference_context_size,
        num_head_joint_cond=num_head_joint_cond, 
        input_size=peva_config['image_size'] // 8, 
        in_channels=4, 
        diffusion_forcing=peva_config.get('diffusion_forcing', 0), 
        is_eval=1, skip_action_embedding=peva_config.get('skip_action_embedding', True), total_feature_dim=peva_config.get('total_feature_dim', None)).to(device)
    
    try:
        model = torch.compile(model)
        print("Compiled model")
    except:
        print("Failed to compile model")
        model = model

    try:
        ckp = torch.load(peva_checkpoint, map_location='cpu', weights_only=False)
        model.load_state_dict(ckp["ema"], strict=True)
    except:
        print(f"Checkpoint {peva_checkpoint} not found. Running with random init.")
        
    model.eval()

    diffusion = create_diffusion(str(diffusion_steps))
    vae = AutoencoderKL.from_pretrained(f"stabilityai/sd-vae-ft-ema").to(device)
    # model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[device])
    # model_without_ddp = model.module
    model_without_ddp = model
    
    peva_stats = {"min": torch.tensor([-2, -1, -1, -1, -1, -1], dtype=torch.float32)[None, None], # 1, 1, 6
                  "max": torch.tensor([2, 1, 1, 1, 1, 1], dtype=torch.float32)[None, None]} # 1, 1, 6
    
    return model, model_without_ddp, diffusion, vae, peva_stats, peva_config

model, model_wo_ddp, diffusion, vae, peva_stats, peva_config = load_peva(PEVA_CONFIG, PEVA_CHECKPOINT, device='cuda')


Compiled model


In [3]:
def get_nymeria_dataset(config, context_size=16-2, split="test"):
    data_config = config["datasets"]["nymeria"]
    
    if "waypoint_spacing" not in data_config:
        data_config["waypoint_spacing"] = 1
    if "negative_goals" not in data_config:
        data_config["negative_goals"] = False
    if "end_slack" not in data_config:
        data_config["end_slack"] = 0
    if "goals_per_obs" not in data_config:
        data_config["goals_per_obs"] = 1

    ### EVAL ONLY -- DO NOT NORMALIZE THE DELTAS
    data_config["normalize"] = False
    
    transform = ([
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    transform = transforms.Compose(transform)
    
    if context_size is None:
        context_size = config["context_size"]
    
    dataset = ViNT_Nymeria_Dataset(
        data_folder=data_config["data_folder"],
        data_split_folder=data_config[split],
        dataset_name="nymeria",
        image_size=config["image_size"],
        transform=transform,
        waypoint_spacing=data_config["waypoint_spacing"],
        min_dist_cat=config["distance"]["min_dist_cat"],
        max_dist_cat=config["distance"]["max_dist_cat"],
        min_action_distance=config["action"]["min_dist_cat"],
        max_action_distance=config["action"]["max_dist_cat"],
        negative_goals=data_config["negative_goals"],
        len_traj_pred=config["len_traj_pred"],
        context_size=context_size,
        goal_type=config.get("goal_type", None),
        preserve_pose_up_down=data_config.get("preserve_pose_up_down", False),
        end_slack=data_config["end_slack"],
        goals_per_obs=data_config["goals_per_obs"],
        normalize=config["normalize"],
        gaussian_normalization_stats_path=data_config["gaussian_normalization_stats_path"],
    )
    return dataset

nomad_config = yaml.load(open(NOMAD_CONFIG), Loader=yaml.FullLoader)
dataset = get_nymeria_dataset(nomad_config)

In [4]:
@torch.no_grad()
def model_forward_wrapper(all_models, curr_obs, curr_delta, latent_size, device, num_cond, rel_t=None, progress=False):
    model, diffusion, vae = all_models
    x = curr_obs.to(device)
    y = curr_delta.to(device)
    
    with torch.amp.autocast('cuda', enabled=True, dtype=torch.bfloat16):
        B, T = x.shape[:2]

        if rel_t is None:
            raise ValueError("Not implemented")

        x = x.flatten(0,1)
        y = y.flatten(0, 1)
        rel_t = rel_t.flatten(0, 1)
        
        x = vae.encode(x).latent_dist.sample().mul_(0.18215).unflatten(0, (B, T))
        x_cond = x[:, :num_cond]
        z = torch.randn(x.shape[0], 4, latent_size, latent_size, device=device)
        t_cond = torch.zeros(x.shape[0], x_cond.shape[1], device=device)
        model_kwargs = dict(y=y, rel_t=rel_t, num_cond=num_cond, x_cond=x_cond, t_cond=t_cond, x_clean=x.flatten(0, 1))
        samples = diffusion.p_sample_loop(model.forward, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=progress, device=device) # B, 4, latent_size, latent_size; samples a single image
        samples = vae.decode(samples / 0.18215).sample # B, 3, image_size, image_size 
        return torch.clip(samples, -1., 1.)

In [5]:
from vint_train.training.nymeria_training_utils import forward_kinematics_wrapper
from scipy.spatial.transform import Rotation as R
import numpy as np


def _compute_pose_and_loss(actn, gt_actn, skel, actn_mask=None):
        gt_xyz, gt_rpy = forward_kinematics_wrapper(gt_actn, skel, XSensConstants.upper_body_num_parts, return_euler=True) # B, num_segments, 3
        pred_xyz, pred_rpy = forward_kinematics_wrapper(actn, skel, XSensConstants.upper_body_num_parts, return_euler=True) # B, num_segments, 3
        res = {}
        for i, body_part_name in enumerate(XSensConstants.part_names[:XSensConstants.upper_body_num_parts]):
            R_gt = R.from_euler('xyz', gt_rpy[:, i, :].detach().cpu().numpy(), degrees=False)
            R_pred = R.from_euler('xyz', pred_rpy[:, i, :].detach().cpu().numpy(), degrees=False)
            ang_dist = torch.from_numpy((R_gt.inv() * R_pred).magnitude() / np.pi * 180).to(actn.device).float() # B
            xyz_dist = torch.norm(gt_xyz[:, i, :] - pred_xyz[:, i, :], dim=-1) # B
            if actn_mask is not None:
                ang_dist = ang_dist * actn_mask
                xyz_dist = xyz_dist * actn_mask
            res[f"{body_part_name}-angular_distance"] = ang_dist
            res[f"{body_part_name}-xyz_distance"] = xyz_dist
        return res

In [6]:
class Preprocessor:
    def __init__(self, transform):
        self.transform = transform

    def transform_obs(self, obs):
        res = {}
        for key in obs:
            if key == "images":
                res[key] = self.transform(obs[key])
            else:
                res[key] = obs[key]
        return res

class WM(torch.nn.Module):
    def __init__(self, peva_model, diffusion, vae, image_size):
        super().__init__()
        self.peva_model = peva_model
        self.diffusion = diffusion
        self.vae = vae
        
        self.image_size = image_size
        self.latent_size = image_size // 8
        
    def encode_obs(self, obs):
        return copy.deepcopy(obs)
    
    def rollout(self, obs_0, act):
        device = act.device
        frames = []
        curr_obs = obs_0['images'] # B, 15, 3, H, W
        for t in range(act.shape[1]):
            x_cond = torch.zeros(curr_obs.shape[0], 15+1, curr_obs.shape[2], curr_obs.shape[3], curr_obs.shape[4], device=device)
            x_cond[:, :15] = curr_obs[:, -15:]
            
            curr_delta = act[:, t:t+1].repeat(1, curr_obs.shape[1], 1,)
            device = curr_obs.device
            
            rel_const = 1. / (64-(-64))  # distance is set in eval, but fixed to [8, 8] for now.
            rel_const = rel_const * 1 # multiply the rel_const by rollout_stride 
            rel_t = (torch.ones(curr_obs.shape[0], 15, device=device) * rel_const)
            
            x_pred = model_forward_wrapper(
                (self.peva_model, self.diffusion, self.vae),
                x_cond,
                curr_delta,
                self.latent_size,
                device,
                num_cond=curr_obs.shape[1],
                rel_t=rel_t,
                progress=True
            )
            x_pred = x_pred[:, None] # B, 1, 3, H, W
            frames.append(x_pred)
            curr_obs = torch.cat([curr_obs[:, 1:], x_pred], dim=1)
        generated_frames = torch.cat(frames, dim=1) # B, T, 3, H, W
        
        all_frames = torch.cat([obs_0['images'], generated_frames], dim=1)
        all_frames = all_frames * 0.5 + 0.5
        for i in range(all_frames.shape[0]):
            image = torch.cat([img for img in all_frames[i]], dim=-1)
            save_image(image, f"rollout_{i}.png")
        return {"images": generated_frames.to(torch.float32)}, None

class ObjectiveFn:
    def __init__(self, device):
        self.device = device
        self.model, self.preprocess = dreamsim(pretrained=True, device=device, cache_dir="/scratch/anw2067/cache")
        
    def __call__(self, pred, goal):
        
        pred_image = pred["images"]
        goal_image = goal["images"]
        B = pred_image.shape[0]
        res = []
        for i in range(B):
            pred_image_pil = transforms.ToPILImage()(pred_image[i, -1])
            goal_image_pil = transforms.ToPILImage()(goal_image[i])
            
            pred_preprocessed = self.preprocess(pred_image_pil).to(self.device)
            goal_preprocessed = self.preprocess(goal_image_pil).to(self.device)

            sim = self.model(pred_preprocessed, goal_preprocessed)
            res.append(sim)
        res = torch.cat(res, dim=0) # B
        print(f"ObjectiveFn: {res.mean().item()}")
        return res

class Evaluator:
    def __init__(self):
        pass
    
    def eval_actions(self, actions_mu, gt_dict):
        """
        actions_mu: B, T, action_dim
        gt_dict: dict of gt_actions, skel
        """
        pred_actions = actions_mu 
        
        deltas_gt = gt_dict["deltas"]
        first_pose = gt_dict["first_pose"]
        xsens_offsets = gt_dict["xsens_offsets"]
        skel = XsensSkeleton(xsens_offsets)
        
        pred_actions = get_action_smpl_torch(first_pose, pred_actions, XSensConstants.upper_body_num_parts) # B, T, 48
        gt_actions = get_action_smpl_torch(first_pose, deltas_gt, XSensConstants.upper_body_num_parts) # B, T, 48
        eval_metrics = _compute_pose_and_loss(pred_actions[:, -1], gt_actions[:, -1], skel)
        
        res = {}
        for k, v in eval_metrics.items():
            if any(x in k.lower() for x in ["head", "hand" "pelvis"]):
                leaf_key = "leaf-" + k.split("-")[1]
                if leaf_key not in res: res[leaf_key] = []
                res[leaf_key].append(v)
        for k, v in res.items():
            res[k] = torch.cat(v, dim=0).mean().item()
            print(f"{k}: {res[k]}")
        for k, v in eval_metrics.items():
            res[k] = v.mean().item()
        return res
        
        
        
cem_planner = CEMPlanner(
    horizon=8,
    topk=2,
    num_samples=64,
    var_scale=1.0,
    opt_steps=16,
    eval_every=1,
    wm=WM(model, diffusion, vae, nomad_config["image_size"][0]),
    action_dim=48,
    objective_fn=ObjectiveFn(device="cuda"),
    preprocessor=Preprocessor(transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])),
    evaluator=Evaluator(),
    wandb_run=None,
    logging_prefix="16top2-var1"
)

wandb.init(project="peva-planning", name="cem")

batch = dataset[0]

obs_images = batch["obs_images"][None] # 1, context_size, 3, H, W
goal_image = batch["goal_image"][None] # 1, 3, H, W
deltas = batch["deltas"][None] # 1, horizon, action_dim
first_pose = batch["first_pose"][None] # 1, 48
xsens_offsets = batch["xsens_offsets"] # 15, 3

obs_0 = {"images": obs_images}
obs_g = {"images": goal_image, "deltas": deltas, "first_pose": first_pose, "xsens_offsets": xsens_offsets}

cem_planner.plan(obs_0, obs_g)

Using cached /scratch/anw2067/cache


Using cache found in /scratch/anw2067/cache/facebookresearch_dino_main
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: alexandernwang. Use `wandb login --relogin` to force relogin
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/wandb/sdk/internal/internal_api.py:12: UserWarning: pkg_resources is deprecated as a

  0%|          | 0/250 [00:00<?, ?it/s]/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()
 18%|█▊        | 45/250 [00:55<04:10,  1.22s/it]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f1f002e2c20>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f1f0033f040, execution_count=6 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f1f0033ed10, raw_cell="class Preprocessor:
    def __init__(self, transfo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/cem.ipynb#W5sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given